# AgentLoop evaluation

Runs the step-level evaluation (base zero-shot, base one-shot, QLoRA) and the error-feedback recovery evaluation, then writes `EVAL.md`.

Needs a GPU runtime and an `HF_TOKEN` Colab secret with access to `google/gemma-4-E2B-it`. Results are written to Google Drive so a disconnected session can resume where it stopped.

In [ ]:
!git clone https://github.com/woyeta/AgentLoop.git 2>/dev/null || echo "Already cloned"
%cd AgentLoop
!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import drive, userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/agentloop_results"
os.makedirs(RESULTS, exist_ok=True)

## 1. Truncation check
Shows how many training examples had their target output cut off by `max_length=256`.

In [ ]:
!python -m evaluation.truncation_check --sample 500 --max-length 256

## 2. Held-out set
Skipped if `evaluation/heldout.json` is already committed.

In [ ]:
if not os.path.exists("evaluation/heldout.json"):
    !python -m evaluation.build_heldout

## 3. Smoke test (3 tasks)

In [ ]:
!python -m evaluation.step_eval --configs qlora --limit 3 --out-dir /content/smoke
!head -c 1500 /content/smoke/step_qlora.jsonl

## 4. Step-level evaluation
Re-run this cell after a disconnect; finished steps are skipped.

In [ ]:
!python -m evaluation.step_eval --configs base base_one_shot qlora --out-dir {RESULTS}

## 5. Recovery from error feedback

In [ ]:
!python -m evaluation.recovery_eval --configs base base_one_shot qlora --out-dir {RESULTS}

## 6. Report

In [ ]:
!python -m evaluation.report --out-dir {RESULTS} --report {RESULTS}/EVAL.md
from IPython.display import Markdown, display
display(Markdown(open(f"{RESULTS}/EVAL.md").read()))